In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev", choices=["dev", "qa", "prd"], label = "Select Environment")
env = dbutils.widgets.get("environment")
print(env)

catalog = f"saleslake_{env}"

bronzeTable = f"{catalog}.bronze_schema_{env}.rawSales"
silverTable = f"{catalog}.silver_schema_{env}.cleanedSales"
print(f"Source  : {bronzeTable}")
print(f"Target  : {silverTable}")

srcFileLoc=f"/Volumes/{catalog}/silver_{env}/vol_saleslake_src_files_{env}/daily_sales/"
# print(srcFileLoc)

In [0]:
spark.sql(f"""
INSERT INTO {silverTable}
SELECT DISTINCT
    CAST(TRIM(sale_id)          AS INTEGER)     AS sale_id,
    CAST(TRIM(product_id)       AS INTEGER)     AS product_id,
    CAST(TRIM(store_id)         AS INTEGER)     AS store_id,
    CAST(TRIM(customer_id)      AS INTEGER)     AS customer_id,
    CAST(TRIM(region_id)        AS INTEGER)     AS region_id,
    CAST(TRIM(quantity)         AS INTEGER)     AS quantity,
    CAST(TRIM(unit_price)       AS DECIMAL(10,2)) AS unit_price,
    CAST(TRIM(gross_amount)     AS DECIMAL(10,2)) AS gross_amount,
    UPPER(TRIM(discount_code))                  AS discount_code,
    CAST(TRIM(discount_amount)  AS DECIMAL(10,2)) AS discount_amount,
    CAST(TRIM(net_amount)       AS DECIMAL(10,2)) AS net_amount,
    CAST(TRIM(cost_amount)      AS DECIMAL(10,2)) AS cost_amount,
    TO_DATE(TRIM(sale_date), 'yyyy-MM-dd')      AS sale_date,       -- e.g. 2025-06-07
    UPPER(TRIM(payment_method))                 AS payment_method,
    UPPER(TRIM(channel))                        AS channel,
    CURRENT_TIMESTAMP()                         AS ingest_ts
FROM {bronzeTable}
WHERE ingest_ts > (
                    SELECT COALESCE(MAX(ingest_ts), TO_DATE('1990-01-01', 'yyyy-MM-dd'))
                    FROM {silverTable}
                  )
ORDER BY CAST(TRIM(sale_id) AS INTEGER)
""")
print("Sales Silver Load Completed Successfully")

In [0]:
%sql
SELECT count(*) FROM saleslake_dev.silver_schema_dev.cleanedSales;

In [0]:
# %sql
# INSERT INTO saleslake_${environment}.silver_schema_${environment}.cleanedSales
# SELECT
#     sale_id,
#     product,
#     category,
#     quantity,
#     price,
#     sale_date,
#     region,
#     ingest_ts
# FROM (
#     SELECT
#         CAST(TRIM(sale_id) AS INTEGER) AS sale_id,
#         UPPER(TRIM(product)) AS product,
#         UPPER(TRIM(category)) AS category,
#         CAST(TRIM(quantity) AS INTEGER) AS quantity,
#         CAST(TRIM(price) AS DOUBLE) AS price,
#         TO_DATE(TRIM(sale_date), 'yyyy-MM-dd') AS sale_date,
#         UPPER(TRIM(region)) AS region,
#         current_timestamp AS ingest_ts,
        
#         ROW_NUMBER() OVER (
#             PARTITION BY TRIM(sale_id)
#             ORDER BY ingest_ts DESC
#         ) AS rn

#     FROM saleslake_${environment}.bronze_schema_${environment}.rawSales
    
#     WHERE ingest_ts > (
#         SELECT COALESCE(
#             MAX(ingest_ts),
#             TO_TIMESTAMP('1900-01-01', 'yyyy-MM-dd')
#         )
#         FROM saleslake_${environment}.silver_schema_${environment}.cleanedSales
#     )
# ) t
# WHERE rn = 1
# ORDER BY sale_id;

In [0]:
# Total Sales Count
spark.sql(f"""
SELECT COUNT(*) AS total_sales
FROM {silverTable}
""").show()

In [0]:
# Duplicate Sale Check
# Each sale_id should appear only once.

spark.sql(f"""
SELECT
    sale_id,
    COUNT(*) AS duplicate_count
FROM {silverTable}
GROUP BY sale_id
HAVING COUNT(*) > 1
""").show()

In [0]:
# Null Key Validation
spark.sql(f"""
SELECT
    SUM(CASE WHEN sale_id IS NULL THEN 1 ELSE 0 END) AS null_sale_id,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
    SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END) AS null_store_id,
    SUM(CASE WHEN region_id IS NULL THEN 1 ELSE 0 END) AS null_region_id
FROM {silverTable}
""").show()

In [0]:
# Negative Amount Validation
# Sales values should never be negative.

spark.sql(f"""
SELECT
    sale_id,
    gross_amount,
    discount_amount,
    net_amount,
    cost_amount
FROM {silverTable}
WHERE gross_amount < 0
OR discount_amount < 0
OR net_amount < 0
OR cost_amount < 0
""").show()

In [0]:
# Invalid Quantity Check
# Quantity must be positive.

spark.sql(f"""
SELECT
    sale_id,
    quantity
FROM {silverTable}
WHERE quantity <= 0
OR quantity IS NULL
""").show()